In [1]:
%cd /drive2/ryusejong/LFF
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "7"
import json 
import time 
import re
import random
import types
import math
import numpy as np 
from tqdm.auto import tqdm
from util.utils import set_seed, read_data, save_result, get_answer_from_text, chat_huggingface, chat_huggingface_with_hidden_states, construct_conversation
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModel


seed = 42
set_seed(seed)

/drive2/ryusejong/LFF


/drive2/ryusejong/miniconda3/envs/llm1/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load prm model (1)
HUGGINGFACE_TOKEN = "XXX"
#prm_path = "UW-Madison-Lee-Lab/Qwen-PRM800K"
prm_path = "UW-Madison-Lee-Lab/Llama-PRM800K"
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda:0


In [3]:
# Load prm model (2)
candidate_tokens = [12, 10]
tokenizer = AutoTokenizer.from_pretrained(prm_path)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    #tokenizer.pad_token_id = tokenizer.eos_token_id
    tokenizer.padding_side = 'left' 
    tokenizer.truncation_side = 'left'
    
prm = AutoModelForCausalLM.from_pretrained(
    prm_path,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    device_map=device
)
prm.generation_config.temperature=None
prm.generation_config.top_p=None
prm.eval()

Loading checkpoint shards: 100%|██████████| 4/4 [00:05<00:00,  1.33s/it]


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((4096,), eps=1e-05)
    (rotary_

In [11]:
# Load output file
output_path = "output/LFF_v9/GSM8K_Llama-3-8B-Instruct_LFF_v9_0_test_512_seed42_portion0.1.jsonl"
output_file = read_data(output_path)[63:65]

print(f"output file: {len(output_file)}")

output file: 2


In [12]:
# Evaluate with prm model
save_path = "reward/Llama-PRM800K/GSM8K_Llama-3-8B-Instruct_zeroshot_CoT_test_512_seed42_portion0.1.jsonl"
reward_data = dict()

for i, data in tqdm(enumerate(output_file), total=len(output_file)):
    # Extract question and reasoning
    question = data["Q1"]["content"].strip()
    reasoning = data["A1"]["content"]
    
    # Split in line
    reasoning_steps = [l.strip() for l in reasoning.split("\n") if l.strip()]
    
    print(f"num_steps: {len(reasoning_steps)}")
    print(reasoning_steps)
    
    # Ceate prefix format for each step
    prm_input = question + ' \n\n' + ' \n\n\n\n'.join(reasoning_steps) + ' \n\n\n\n'
    
    # Get reward
    with torch.no_grad():
        inputs = torch.tensor([tokenizer.encode(prm_input)]).to(device)
        logits = prm(inputs).logits[:,:,candidate_tokens]
        print(logits.shape)
        scores = logits.softmax(dim=-1)[:,:,1]
        print(scores.shape)
        step_scores = scores[inputs == 23535]
        step_probs  = step_scores.tolist()
        print(step_probs)
    
    """
    # Save the reward
    reward_data = {
        "index": data["index"],
        "question": question,
        "reasoning_steps": reasoning_steps,
        "step_probs": step_probs,
        "pred_ans": data["pred_ans"],
        "answer": data["answer"]
    }
    save_result(reward_data, save_path)
    """
    

 50%|█████     | 1/2 [00:00<00:00,  6.83it/s]

num_steps: 15
["Let's break this problem down step-by-step!", "1. We know that it costs $194 per meter to repave a street. This means that the cost of repaving Monica's street, which is 150 meters long, is:", "Cost of repaving Monica's street = Number of meters x Cost per meter", '= 150 meters x $194 per meter', '= $28,700', "2. Now, we need to find the cost of repaving Lewis' street, which is 490 meters long. We can use the same formula:", "Cost of repaving Lewis' street = Number of meters x Cost per meter", '= 490 meters x $194 per meter', '= $95,240', "3. To find out how much more it costs to repave Lewis' street, we need to subtract the cost of repaving Monica's street from the cost of repaving Lewis' street:", "Cost difference = Cost of repaving Lewis' street - Cost of repaving Monica's street", '= $95,240 - $28,700', '= $66,540', '## 66,540 ##', "So, it costs $66,540 more to repave Lewis' street, which is 490 meters long, compared to repaving Monica's street, which is 150 meters 

100%|██████████| 2/2 [00:00<00:00,  7.16it/s]

torch.Size([1, 265, 2])
torch.Size([1, 265])
[0.94921875, 0.91796875, 0.9765625, 0.98046875, 0.98828125, 0.9765625, 0.94921875, 0.984375, 0.92578125]
